# Cross-Seed Invariance Analysis

This notebook provides interactive comparison of training runs for seed invariance analysis—answering the question: **Are detected events real, or artifacts of randomness?**

## Purpose

Events that appear consistently across runs with different random seeds indicate genuine learning dynamics; events appearing in only one seed are likely noise. This notebook computes invariance metrics at multiple levels and provides visualizations to diagnose reproducibility.

## The Invariance Stack

The analysis computes invariance at multiple levels, forming a diagnostic hierarchy:

| Level | Metric | Question |
|-------|--------|----------|
| **Signal** | Trajectory Correlation | Do the geometry curves look the same? |
| **Neighborhood** | Neighbor Coverage | Do both seeds show activity in the same neighborhoods? |
| **Peak** | IoU Jaccard | Do they select the same discrete event winners? |

### Common Pattern: "Stable Energy Landscape / Unstable Argmax"

The most typical result:
- **High trajectory correlation** (~0.91) — The underlying signal is reproducible
- **Moderate neighbor coverage** (~50%) — Events occur in the same neighborhoods
- **Low peak IoU Jaccard** (~29%) — Different discrete winners are selected

This means: *Both seeds detect the same learning dynamics, but pick different winners within matched neighborhoods due to suppression sensitivity.*

## Key Metrics Explained

### Neighbor Coverage
Measures whether events have neighbors in the other run within suppression radius. **Not a true Jaccard**—it's a symmetric "has-neighbor" score:
```
Formula: (A_has_neighbor + B_has_neighbor) / (|A| + |B|)
```

### Peak IoU Jaccard  
Traditional event matching using temporal window overlap. Sensitive to window size and grid discretization.

### Winner Stability
Within matched neighborhoods, do both runs select the same peak (by argmax score)?

## Sections

1. **Load Data** — Load runs and verify configuration match
2. **Event Raster Plots** — Visual comparison of event locations
3. **Common Events** — Strictly matching events (±5 steps)
4. **IoU Invariance** — Jaccard curves vs threshold
5. **Grid Cadence Analysis** — Window/suppression scale mismatch diagnosis
6. **Center Distance Diagnostics** — Event alignment histogram
7. **Window Dilation Experiment** — IoU ceiling investigation
8. **Neighbor Coverage Analysis** — The "middle layer" metric
9. **Candidate vs Selected** — Pre/post suppression comparison
10. **IoU Distribution** — Histogram of match qualities
11. **Trajectory Comparison** — Per-layer correlation plots
12. **Composite Events** — Multi-metric event invariance
13. **Phase Distribution** — Temporal event distribution
14. **Retention Analysis** — Detection pipeline comparison
15. **Summary Statistics** — Overall invariance assessment
16. **LLM Synthesis** — AI-assisted qualitative analysis

## LLM Analysis

This notebook includes **automatic LLM analysis** for each major section. After each visualization, an LLM reviews the data and provides:
- **Interpretation**: What the data shows
- **Key observations**: Notable patterns or anomalies
- **Questions**: Follow-up questions for investigation

Toggle `ENABLE_LLM_ANALYSIS = False` in the configuration cell to disable LLM calls.

## Documentation

See `squiggle-matching/docs/cross_seed_analysis.md` for comprehensive documentation of the cross-seed analysis methodology.

In [ ]:
# Configuration - Set your run IDs here (2 or more runs)
RUN_IDS = [
    "20260129_210111_research_test_a1_s37",
    "20260129_145609_research_test_a1_s42",
    # Add more run IDs here for N-way comparison:
    # "20260129_HHMMSS_research_test_a1_s123",
]

# Optional: specify analysis_id for each run (None = auto-select latest)
# Must be same length as RUN_IDS or None
ANALYSIS_IDS = None  # or: [None, None, "w10_p1_s50"]

# Matching parameters
STEP_TOLERANCE = 5
IOU_THRESHOLD = 0.2

# LLM Analysis Configuration
ENABLE_LLM_ANALYSIS = True  # Set to False to skip LLM calls
LLM_BACKEND = "openai"  # "openai" or "anthropic"
LLM_MODEL = "gpt-4o-mini"  # Use mini for speed/cost, or "gpt-4o" for best quality

# Storage for section analyses (used for overall synthesis at the end)
SECTION_ANALYSES = []

In [ ]:
# Force reimport of modified modules (run this after code changes)
import importlib
import squiggle_analysis.compare_runs
importlib.reload(squiggle_analysis.compare_runs)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import combinations

from squiggle_analysis.compare_runs import (
    load_run_data,
    find_common_events,
    compute_event_jaccard,
    compute_window_overlap_invariance,
    compute_invariance_curve,
    compute_composite_invariance,
    compute_neighbor_coverage,  # Bidirectional neighbor coverage metric
    compute_trajectory_correlation,
    analyze_event_phases,
)
from squiggle_analysis.trajectories import extract_metric_trajectories

# LLM analysis imports (for per-section analysis)
from squiggle_analysis.llm_analysis import (
    analyze_raster_plot,
    analyze_common_events,
    analyze_iou_invariance,
    analyze_iou_distribution,
    analyze_trajectory_comparison,
    analyze_composite_events,
    analyze_phase_distribution,
    analyze_retention,
    analyze_summary,
    analyze_overall,
    display_analysis,
)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

print("Imports OK")

## 1. Load Run Data

In [ ]:
# Load all runs
analysis_ids = ANALYSIS_IDS if ANALYSIS_IDS else [None] * len(RUN_IDS)
runs = [load_run_data(run_id, aid) for run_id, aid in zip(RUN_IDS, analysis_ids)]

print(f"Loaded {len(runs)} runs:\n")
for i, run in enumerate(runs):
    print(f"Run {i+1}: {run.run_id}")
    print(f"  - Analysis ID: {run.analysis_id}")
    print(f"  - Events: {len(run.events_df)}")
    print(f"  - Seed: {run.meta.get('seed', '?')}")
    print()

# Extract seeds for later use
run_seeds = [run.meta.get('seed', '?') for run in runs]

## 2. Event Raster Plots

Visualize event locations across layers and steps.

In [ ]:
def plot_event_raster(events_df, title, ax=None, max_layer=24):
    """Plot event raster (step x layer, colored by metric)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 6))
    
    # Color map for metrics
    metric_colors = {
        "effective_rank": "#1f77b4",
        "sv_entropy": "#2ca02c",
        "topk_mass_k8": "#ff7f0e",
        "__composite__": "#d62728",
    }
    
    # Plot single-metric events
    for metric, color in metric_colors.items():
        if metric == "__composite__":
            continue
        mask = events_df["metric"] == metric
        if mask.sum() > 0:
            ax.scatter(
                events_df.loc[mask, "step"],
                events_df.loc[mask, "layer"],
                c=color,
                s=40,
                alpha=0.7,
                label=metric,
                marker='o'
            )
    
    # Plot composites with different marker
    comp_mask = events_df["event_type"] == "change_point_composite"
    if comp_mask.sum() > 0:
        ax.scatter(
            events_df.loc[comp_mask, "step"],
            events_df.loc[comp_mask, "layer"],
            c=metric_colors["__composite__"],
            s=100,
            alpha=0.8,
            label="composite",
            marker='s'
        )
    
    ax.set_xlabel("Step", fontsize=12)
    ax.set_ylabel("Layer", fontsize=12)
    ax.set_ylim(-0.5, max_layer - 0.5)
    ax.set_title(title, fontsize=14)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)
    
    return ax

In [ ]:
# Plot raster for each run
n_runs = len(runs)
fig, axes = plt.subplots(n_runs, 1, figsize=(14, 5 * n_runs), sharex=True)
if n_runs == 1:
    axes = [axes]

for i, run in enumerate(runs):
    seed = run.meta.get('seed', '?')
    short_id = run.run_id[-25:] if len(run.run_id) > 25 else run.run_id
    plot_event_raster(run.events_df, f"Run {i+1}: {short_id} (seed {seed})", axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# LLM Analysis: Raster Plots
if ENABLE_LLM_ANALYSIS:
    def get_layer_range(events_df):
        if events_df.empty or "layer" not in events_df.columns:
            return "N/A"
        return f"{events_df['layer'].min()}-{events_df['layer'].max()}"
    
    def get_step_range(events_df):
        if events_df.empty or "step" not in events_df.columns:
            return "N/A"
        return f"{events_df['step'].min()}-{events_df['step'].max()}"
    
    # Build run info for all runs
    run_infos = []
    for run in runs:
        run_infos.append({
            "run_id": run.run_id,
            "seed": run.meta.get("seed", "?"),
            "n_events": len(run.events_df),
            "layer_range": get_layer_range(run.events_df),
            "step_range": get_step_range(run.events_df),
        })
    
    # For N runs, we pass the first two to the existing function but provide context for all
    # Build a custom analysis for N runs
    if len(runs) == 2:
        raster_analysis = analyze_raster_plot(
            run_a_info=run_infos[0],
            run_b_info=run_infos[1],
            backend=LLM_BACKEND,
            model=LLM_MODEL,
        )
    else:
        # For N>2 runs, use a custom prompt with all run info
        from squiggle_analysis.llm_analysis.section_analyzer import _make_section_request
        
        section_prompt = """Analyze these event raster plots showing detected change points across layers and steps for multiple runs.

Consider:
- Are events clustered in specific layers or distributed across all runs?
- Are there temporal patterns (early burst, late clustering) consistent across runs?
- Do the runs show similar spatial patterns?
- Which runs are most similar/different to each other?
- Any obvious asymmetries between runs?"""
        
        runs_context = "\n".join([
            f"Run {i+1}: {info['run_id'][:30]} (seed={info['seed']})\n"
            f"  - Total events: {info['n_events']}\n"
            f"  - Layer range: {info['layer_range']}\n"
            f"  - Step range: {info['step_range']}"
            for i, info in enumerate(run_infos)
        ])
        
        raster_analysis = _make_section_request(
            "raster_plot_multi",
            section_prompt,
            runs_context,
            backend=LLM_BACKEND,
            model=LLM_MODEL,
        )
    
    display_analysis(raster_analysis, "Raster Plots")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Raster Plots",
        "interpretation": raster_analysis.interpretation,
        "observations": raster_analysis.key_observations,
    })

## 3. Common Events Overlay

Show events that appear in both runs.

In [ ]:
# Find common events
common_events = find_common_events(runs, step_tolerance=STEP_TOLERANCE)
print(f"Found {len(common_events)} common events (strict matching, ±{STEP_TOLERANCE} steps)")

if not common_events.empty:
    display(common_events.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

if not common_events.empty:
    # Color by metric
    metric_colors = {
        "effective_rank": "#1f77b4",
        "sv_entropy": "#2ca02c",
        "topk_mass_k8": "#ff7f0e",
    }
    
    for metric, color in metric_colors.items():
        mask = common_events["metric"] == metric
        if mask.sum() > 0:
            ax.scatter(
                common_events.loc[mask, "mean_step"],
                common_events.loc[mask, "layer"],
                c=color,
                s=common_events.loc[mask, "mean_score"] * 5,  # Size by score
                alpha=0.7,
                label=f"{metric} ({mask.sum()})"
            )
    
    # Add error bars for step variance
    ax.errorbar(
        common_events["mean_step"],
        common_events["layer"],
        xerr=common_events["std_step"],
        fmt='none',
        ecolor='gray',
        alpha=0.3,
        capsize=2
    )

ax.set_xlabel("Mean Step", fontsize=12)
ax.set_ylabel("Layer", fontsize=12)
ax.set_title(f"Common Events (n={len(common_events)}) - Size = Score", fontsize=14)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# LLM Analysis: Common Events
if ENABLE_LLM_ANALYSIS:
    # Build summary of common events distribution
    common_summary = {}
    if not common_events.empty:
        if "layer" in common_events.columns:
            common_summary["layer_counts"] = common_events["layer"].value_counts().head(5).to_dict()
        if "metric" in common_events.columns:
            common_summary["metric_counts"] = common_events["metric"].value_counts().to_dict()
    
    total_events = sum(len(run.events_df) for run in runs)
    
    common_analysis = analyze_common_events(
        n_common=len(common_events),
        step_tolerance=STEP_TOLERANCE,
        common_events_summary=common_summary,
        run_a_total=len(runs[0].events_df),
        run_b_total=len(runs[1].events_df) if len(runs) > 1 else 0,
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(common_analysis, "Common Events")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Common Events",
        "interpretation": common_analysis.interpretation,
        "observations": common_analysis.key_observations,
    })

## 4. IoU Invariance Analysis

In [ ]:
# Compute pairwise window overlap invariance for all run pairs
pairwise_overlap_results = {}
run_pairs = list(combinations(range(len(runs)), 2))

print(f"Computing pairwise IoU metrics for {len(run_pairs)} pairs...\n")

for i, j in run_pairs:
    pair_key = f"{i+1}-{j+1}"
    pair_runs = [runs[i], runs[j]]
    overlap_result = compute_window_overlap_invariance(pair_runs, iou_threshold=IOU_THRESHOLD, compute_all_ious=True)
    pairwise_overlap_results[pair_key] = overlap_result
    
    seed_i = runs[i].meta.get('seed', '?')
    seed_j = runs[j].meta.get('seed', '?')
    print(f"Pair {pair_key} (seed {seed_i} vs {seed_j}):")
    print(f"  - Jaccard (IoU >= {IOU_THRESHOLD}): {overlap_result['jaccard_overlap']:.1%}")
    print(f"  - Matched pairs: {overlap_result['n_matched_pairs']}")
    print(f"  - Mean IoU: {overlap_result['mean_iou']:.3f}")
    print()

# Compute aggregate statistics
all_jaccards = [r['jaccard_overlap'] for r in pairwise_overlap_results.values()]
print(f"Aggregate across {len(run_pairs)} pairs:")
print(f"  - Mean Jaccard: {np.mean(all_jaccards):.1%}")
print(f"  - Std Jaccard: {np.std(all_jaccards):.1%}")
print(f"  - Min Jaccard: {np.min(all_jaccards):.1%}")
print(f"  - Max Jaccard: {np.max(all_jaccards):.1%}")

# Use first pair for backward compatibility with subsequent cells
overlap_result = list(pairwise_overlap_results.values())[0] if pairwise_overlap_results else {}

In [ ]:
# Compute invariance curve for each pair (or just the first pair if many)
# Use first pair for the detailed curve
if len(runs) == 2:
    curve_result = compute_invariance_curve(runs, iou_thresholds=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45])
else:
    # For N>2, compute curve for first pair and note it's representative
    curve_result = compute_invariance_curve([runs[0], runs[1]], iou_thresholds=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45])
    print(f"Note: Curve shown for pair 1-2 (seeds {run_seeds[0]} vs {run_seeds[1]}) as representative\n")

curve_data = curve_result.get("curve", [])

if curve_data:
    curve_df = pd.DataFrame(curve_data)
    display(curve_df)

In [ ]:
# Plot IoU invariance curve
if curve_data:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Jaccard curve
    ax = axes[0]
    ax.plot(curve_df["iou_threshold"], curve_df["jaccard"] * 100, 'o-', linewidth=2, markersize=8)
    ax.set_xlabel("IoU Threshold (τ)", fontsize=12)
    ax.set_ylabel("Jaccard (%)", fontsize=12)
    ax.set_title("Jaccard vs IoU Threshold", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(curve_df["jaccard"] * 100) * 1.1)
    
    # Matched count curve
    ax = axes[1]
    ax.plot(curve_df["iou_threshold"], curve_df["n_matched"], 's-', linewidth=2, markersize=8, color='green')
    ax.set_xlabel("IoU Threshold (τ)", fontsize=12)
    ax.set_ylabel("Matched Pairs", fontsize=12)
    ax.set_title("Matched Pairs vs IoU Threshold", fontsize=14)
    ax.grid(True, alpha=0.3)
    
    # Mean IoU of matches
    ax = axes[2]
    valid_mask = curve_df["mean_iou"] > 0
    ax.plot(curve_df.loc[valid_mask, "iou_threshold"], curve_df.loc[valid_mask, "mean_iou"], '^-', linewidth=2, markersize=8, color='orange')
    ax.set_xlabel("IoU Threshold (τ)", fontsize=12)
    ax.set_ylabel("Mean IoU of Matches", fontsize=12)
    ax.set_title("Match Quality vs Threshold", fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# LLM Analysis: IoU Invariance
if ENABLE_LLM_ANALYSIS:
    # For N>2 runs, provide aggregate context
    if len(runs) > 2:
        # Modify overlap_result to include aggregate stats
        overlap_result_with_agg = dict(overlap_result)
        overlap_result_with_agg['n_pairs_compared'] = len(pairwise_overlap_results)
        overlap_result_with_agg['mean_jaccard_across_pairs'] = np.mean(all_jaccards)
        overlap_result_with_agg['std_jaccard_across_pairs'] = np.std(all_jaccards)
    else:
        overlap_result_with_agg = overlap_result
    
    iou_analysis = analyze_iou_invariance(
        overlap_result=overlap_result_with_agg,
        curve_data=curve_data if curve_data else [],
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(iou_analysis, "IoU Invariance")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "IoU Invariance",
        "interpretation": iou_analysis.interpretation,
        "observations": iou_analysis.key_observations,
    })

### 4.1 Center-Distance Diagnostics

For matched pairs, how far apart are the event centers in steps?

### 4.0 Grid Cadence & Window Scale Analysis

Understanding the relationship between capture cadence, event windows, and suppression radius.

In [ ]:
# Grid Cadence & Window Scale Analysis
# This helps understand why IoU has a ceiling and how to interpret it

# Get the step grid from geometry
if runs:
    run = runs[0]
    steps = sorted(run.geometry_df["step"].unique())
    step_deltas = np.diff(steps)
    
    print("Step Grid Analysis:")
    print(f"  - Grid size: {len(steps)} capture points")
    print(f"  - Step range: [{steps[0]}, {steps[-1]}]")
    print(f"  - Median cadence: {np.median(step_deltas):.1f} steps between captures")
    print(f"  - Min/Max cadence: {np.min(step_deltas):.0f} / {np.max(step_deltas):.0f}")
    print()
    
    # Detection parameters (from detection code defaults)
    event_window_radius = 1  # capture points (default)
    suppression_radius_steps = 15  # step units (default)
    
    cadence = np.median(step_deltas)
    
    print("Scale Mismatch Diagnosis:")
    print(f"  - event_window_radius = {event_window_radius} capture points")
    print(f"    -> Window width = {2*event_window_radius + 1} capture points = ~{(2*event_window_radius + 1) * cadence:.0f} steps")
    print(f"  - suppression_radius_steps = {suppression_radius_steps} steps")
    print(f"    -> Suppression radius = ~{suppression_radius_steps / cadence:.1f} capture points")
    print()
    
    # Theoretical IoU ceiling calculation
    # With radius=1, windows are 3 capture points wide
    # If events are offset by 1 tick: overlap=2, union=4 -> IoU=0.5
    # But in step-space with cadence, it's worse due to boundary semantics
    
    window_width_points = 2 * event_window_radius + 1
    for offset in [0, 1, 2]:
        overlap = max(0, window_width_points - offset)
        union = window_width_points + offset
        iou = overlap / union if union > 0 else 0
        print(f"  - IoU for {offset}-tick offset: {iou:.3f} (grid-point calculation)")
    
    print()
    print("Recommendations:")
    if event_window_radius == 1:
        print("  - event_window_radius=1 is VERY small (3 capture points)")
        print("    -> IoU ceiling is artificially low (~0.5 for 1-tick offset)")
        print("    -> Consider increasing to 2 or 3 for more realistic overlap")
    
    supp_in_points = suppression_radius_steps / cadence
    if supp_in_points >= 3:
        print(f"  - suppression_radius ({suppression_radius_steps} steps) = ~{supp_in_points:.1f} capture points")
        print("    -> Wide suppression + tiny windows = selection instability")
        print("    -> Consider using index-based suppression for cadence-independence")
    
    # Show first 20 steps for reference
    print()
    print(f"First 20 steps: {steps[:20]}")

In [ ]:
# Center-Distance Diagnostics: How far apart are matched event centers?
# This tests the "1-2 tick offset" hypothesis

def compute_center_distances(overlap_result):
    """Extract center-to-center step distances from matched pairs."""
    matches = overlap_result.get("matches", [])
    if not matches:
        return []
    return [abs(m["step_a"] - m["step_b"]) for m in matches]

# Collect all center distances across pairs
all_center_distances = []
for pair_key, result in pairwise_overlap_results.items():
    # Use tau=0 to get all matches regardless of IoU
    full_overlap = compute_window_overlap_invariance(
        [runs[int(pair_key.split('-')[0])-1], runs[int(pair_key.split('-')[1])-1]], 
        iou_threshold=0.0, 
        compute_all_ious=True
    )
    distances = compute_center_distances(full_overlap)
    all_center_distances.extend(distances)

if all_center_distances:
    distances_arr = np.array(all_center_distances)
    
    print("Center-Distance Statistics (matched pairs at tau=0):")
    print(f"  - n pairs: {len(distances_arr)}")
    print(f"  - Mean |centerA - centerB|: {np.mean(distances_arr):.1f} steps")
    print(f"  - Median: {np.median(distances_arr):.1f} steps")
    print(f"  - Std: {np.std(distances_arr):.1f} steps")
    print(f"  - p90: {np.percentile(distances_arr, 90):.1f} steps")
    print(f"  - p95: {np.percentile(distances_arr, 95):.1f} steps")
    print(f"  - Max: {np.max(distances_arr):.1f} steps")
    
    # Plot histogram
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(distances_arr, bins=min(30, len(set(distances_arr))), edgecolor='black', alpha=0.7)
    ax.axvline(np.median(distances_arr), color='orange', linestyle='--', linewidth=2, label=f'median={np.median(distances_arr):.1f}')
    ax.axvline(np.percentile(distances_arr, 90), color='red', linestyle=':', linewidth=2, label=f'p90={np.percentile(distances_arr, 90):.1f}')
    ax.set_xlabel("Center-to-Center Distance (steps)", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_title(f"Event Center Alignment (n={len(distances_arr)} matched pairs)", fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Interpret
    if np.median(distances_arr) <= 5:
        print("\n-> Median offset is small - events are well-aligned despite low IoU.")
        print("   The IoU ceiling is likely due to SHORT WINDOWS, not misalignment.")
    else:
        print("\n-> Median offset is larger - events may genuinely differ in timing.")
else:
    print("No matched pairs found.")

### 4.2 IoU Dilation Experiment

What happens if we expand event windows by ±1 tick before computing IoU? This models the hypothesis that events are "off by 1-2 grid points" due to discretization.

In [ ]:
# IoU Dilation Experiment: Use built-in window_dilation parameter
# This tests whether the IoU ceiling is due to discretization/grid alignment

if len(runs) >= 2:
    print("IoU Dilation Experiment (first pair):\n")
    print("Using built-in window_dilation parameter to expand comparison windows\n")
    
    # Baseline (no dilation)
    baseline = compute_window_overlap_invariance(
        [runs[0], runs[1]], 
        iou_threshold=IOU_THRESHOLD,
        compute_all_ious=True,
        window_dilation=0
    )
    
    # Dilate by 1 step (expands windows by ±1 step)
    dilated_1 = compute_window_overlap_invariance(
        [runs[0], runs[1]], 
        iou_threshold=IOU_THRESHOLD,
        compute_all_ious=True,
        window_dilation=1
    )
    
    # Dilate by 2 steps
    dilated_2 = compute_window_overlap_invariance(
        [runs[0], runs[1]], 
        iou_threshold=IOU_THRESHOLD,
        compute_all_ious=True,
        window_dilation=2
    )
    
    # Dilate by cadence (one full tick)
    cadence = int(np.median(np.diff(sorted(runs[0].geometry_df["step"].unique()))))
    dilated_cadence = compute_window_overlap_invariance(
        [runs[0], runs[1]], 
        iou_threshold=IOU_THRESHOLD,
        compute_all_ious=True,
        window_dilation=cadence
    )
    
    print(f"{'Metric':<20} {'Baseline':<12} {'±1 step':<12} {'±2 steps':<12} {'±{0} (cadence)':<12}".format(cadence))
    print("-" * 68)
    print(f"{'Jaccard':<20} {baseline['jaccard_overlap']:.1%}{'':>5} {dilated_1['jaccard_overlap']:.1%}{'':>5} {dilated_2['jaccard_overlap']:.1%}{'':>5} {dilated_cadence['jaccard_overlap']:.1%}")
    print(f"{'Matched pairs':<20} {baseline['n_matched_pairs']:<12} {dilated_1['n_matched_pairs']:<12} {dilated_2['n_matched_pairs']:<12} {dilated_cadence['n_matched_pairs']}")
    print(f"{'Mean IoU':<20} {baseline['mean_iou']:.3f}{'':>5} {dilated_1['mean_iou']:.3f}{'':>5} {dilated_2['mean_iou']:.3f}{'':>5} {dilated_cadence['mean_iou']:.3f}")
    
    # Get max IoU from each
    max_baseline = baseline.get('iou_distribution', {}).get('max', 0)
    max_d1 = dilated_1.get('iou_distribution', {}).get('max', 0)
    max_d2 = dilated_2.get('iou_distribution', {}).get('max', 0)
    max_dc = dilated_cadence.get('iou_distribution', {}).get('max', 0)
    print(f"{'Max IoU':<20} {max_baseline:.3f}{'':>5} {max_d1:.3f}{'':>5} {max_d2:.3f}{'':>5} {max_dc:.3f}")
    
    # Interpret
    jaccard_gain = dilated_cadence['jaccard_overlap'] - baseline['jaccard_overlap']
    if jaccard_gain > 0.05:
        print(f"\n-> Dilation by ±{cadence} steps (1 tick) improves Jaccard by {jaccard_gain:.1%}")
        print("   This confirms the IoU ceiling is due to discretization, not misalignment.")
    elif jaccard_gain > 0.02:
        print(f"\n-> Dilation has moderate effect (+{jaccard_gain:.1%}) - some discretization impact.")
    else:
        print(f"\n-> Dilation has minimal effect ({jaccard_gain:+.1%}) - windows are already well-aligned.")

### 4.2.1 Neighbor Coverage Analysis

The "middle layer" metric: measures whether events have neighbors within suppression radius scale.

**Key distinction:**
- **Neighbor Coverage** answers: "Do both seeds show activity in the same neighborhoods?"
- **Peak/IoU Jaccard** answers: "Do they select the same discrete winners?"

**Definition:**
- **Signature (matching key)**: (layer, metric, event_type)
- **Neighbor condition**: ∃ event in other run with |Δstep| ≤ radius_steps
- **Formula**: (A_has_neighbor + B_has_neighbor) / (|A| + |B|)

**This is NOT a true Jaccard** (no set intersection/union). It's a symmetric coverage score.

**Interpretation:**
- High coverage + low Event Jaccard = "stable energy landscape, unstable argmax" (regions are consistent, winner selection jitters)
- Low coverage = genuinely different event locations

In [ ]:
# Neighbor Coverage Analysis - measures bidirectional neighbor presence at suppression radius scale
# This is the "middle layer" between trajectories and selected peaks

if len(runs) >= 2:
    print("Neighbor Coverage Analysis")
    print("=" * 50)
    print("Neighbor Coverage: 'Do both seeds show activity in the same neighborhoods?'")
    print("Peak/IoU Jaccard: 'Do they select the same discrete winners?'")
    print()
    print(f"Radius: ±{15} steps (= suppression_radius)")
    print("Winner = argmax by score (tiebreak: smallest step)")
    print()
    
    # Compute neighbor coverage for all pairs
    for pair_key in pairwise_overlap_results.keys():
        i, j = [int(x)-1 for x in pair_key.split('-')]
        nc_result = compute_neighbor_coverage([runs[i], runs[j]], radius_steps=15)
        
        seed_i = runs[i].meta.get('seed', '?')
        seed_j = runs[j].meta.get('seed', '?')
        
        print(f"Pair {pair_key} (seed {seed_i} vs {seed_j}):")
        print(f"  Neighbor Coverage (±15): {nc_result['neighbor_coverage']:.1%}")
        print(f"    A has neighbor: {nc_result['coverage_a']:.1%} ({nc_result['events_a_with_neighbor']}/{nc_result['total_events_a']})")
        print(f"    B has neighbor: {nc_result['coverage_b']:.1%} ({nc_result['events_b_with_neighbor']}/{nc_result['total_events_b']})")
        print(f"  Signatures with overlap: {nc_result['n_signatures_with_overlap']}")
        print(f"  Winner stability (exact match): {nc_result['winner_stability']:.1%}")
        print(f"  Winner close stability (±{nc_result['radius_steps']//2} steps): {nc_result['winner_close_stability']:.1%}")
        
        # Display NN distance stats
        nn_stats = nc_result.get('nn_distance_stats', {})
        if nn_stats:
            print(f"  Nearest-neighbor distance (A↔B):")
            print(f"    Median: {nn_stats['median']:.1f} steps")
            print(f"    p90: {nn_stats['p90']:.1f} steps, p95: {nn_stats['p95']:.1f} steps")
            print(f"    Within radius (±{nc_result['radius_steps']}): {nn_stats['within_radius']:.1%}")
            print(f"    Within half-radius (±{nc_result['radius_steps']//2}): {nn_stats['within_half_radius']:.1%}")
        
        print(f"  → {nc_result['interpretation']}")
        print()
    
    # Compare to event-level Jaccard
    if len(pairwise_overlap_results) > 0:
        first_pair_key = list(pairwise_overlap_results.keys())[0]
        event_jaccard = pairwise_overlap_results[first_pair_key]['jaccard_overlap']
        nc_result = compute_neighbor_coverage([runs[0], runs[1]], radius_steps=15)
        neighbor_coverage = nc_result['neighbor_coverage']
        
        print("=" * 50)
        print("COMPARISON (first pair):")
        print(f"  Event Jaccard (IoU≥{IOU_THRESHOLD}): {event_jaccard:.1%}")
        print(f"  Neighbor Coverage (±15): {neighbor_coverage:.1%}")
        
        if neighbor_coverage > event_jaccard * 1.5:
            print()
            print("  → Neighbor Coverage >> Event Jaccard")
            print("    Both seeds detect events in the same NEIGHBORHOODS,")
            print("    but pick different WINNERS within those neighborhoods.")
            print("    This is 'stable energy landscape / unstable argmax'.")

### 4.3 Candidate vs Selected Overlap

Are events being filtered out by suppression, or do they genuinely differ? Compare pre-suppression (candidate-level) vs post-suppression (selected-level) overlap.

In [ ]:
# Candidate vs Selected Overlap Analysis
# Uses detection_summary to compare raw candidates vs selected events

from squiggle_core import paths as core_paths

def load_detection_summary(run):
    """Load detection summary for a run (contains candidate counts per series)."""
    summary_path = core_paths.detection_summary_path(run.run_id, run.analysis_id)
    if summary_path.exists():
        return pd.read_parquet(summary_path)
    return None

# Load detection summaries
detection_summaries = [load_detection_summary(run) for run in runs]
has_summaries = all(ds is not None for ds in detection_summaries)

if has_summaries and len(runs) >= 2:
    print("Candidate vs Selected Overlap Analysis:\n")
    
    # Display retention stats side by side
    for i, (run, ds) in enumerate(zip(runs, detection_summaries)):
        seed = run.meta.get('seed', '?')
        n_candidates = ds["n_candidates_raw"].sum()
        n_selected = ds["n_selected_final"].sum()
        n_suppressed = ds["n_skipped_suppression"].sum()
        n_topk = ds["n_skipped_topk"].sum()
        
        print(f"Run {i+1} (seed {seed}):")
        print(f"  - Candidates (raw): {n_candidates}")
        print(f"  - Selected (final): {n_selected}")
        print(f"  - Skipped (suppression): {n_suppressed}")
        print(f"  - Skipped (top-k): {n_topk}")
        print(f"  - Retention: {n_selected/max(n_candidates,1):.1%}")
        print()
    
    # Compute what fraction of candidates are shared vs what fraction of selected are shared
    # This requires comparing candidate-level events, which we can approximate from detection_summary
    
    # For now, show the retention gap interpretation
    r1, r2 = runs[0].retention, runs[1].retention
    if r1 and r2:
        print("Interpretation:")
        
        # Suppression comparison
        supp_rate_1 = (r1.suppression_pre + r1.suppression_post) / max(r1.n_candidates, 1)
        supp_rate_2 = (r2.suppression_pre + r2.suppression_post) / max(r2.n_candidates, 1)
        
        print(f"  - Suppression rate: {supp_rate_1:.1%} (run 1) vs {supp_rate_2:.1%} (run 2)")
        
        if abs(supp_rate_1 - supp_rate_2) > 0.1:
            print("  -> Suppression rates differ significantly - selection instability likely.")
        else:
            print("  -> Suppression rates are similar - events filtered consistently.")
        
        # Check if candidate density explains differences
        density_ratio = r1.mean_candidates_post_per_series / max(r2.mean_candidates_post_per_series, 0.1)
        if abs(density_ratio - 1.0) > 0.3:
            print(f"  -> Candidate density differs {density_ratio:.1f}x - more candidates = more collisions.")
else:
    print("Detection summaries not available for candidate-level analysis.")
    print("Re-run event detection with recent squiggle_analysis to generate detection_summary.parquet.")

## 5. IoU Distribution Histogram

In [ ]:
# Collect all IoUs from all pairwise comparisons
all_match_ious = []
for pair_key, result in pairwise_overlap_results.items():
    full_overlap = compute_window_overlap_invariance(
        [runs[int(pair_key.split('-')[0])-1], runs[int(pair_key.split('-')[1])-1]], 
        iou_threshold=0.0, 
        compute_all_ious=True
    )
    pair_ious = [m["iou"] for m in full_overlap.get("matches", [])]
    all_match_ious.extend(pair_ious)

if all_match_ious:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.hist(all_match_ious, bins=30, edgecolor='black', alpha=0.7)
    
    # Add vertical lines for key thresholds
    for tau, color in [(0.2, 'green'), (0.3, 'orange'), (0.4, 'red')]:
        ax.axvline(tau, color=color, linestyle='--', linewidth=2, label=f'tau={tau}')
    
    # Add max IoU line
    max_iou = max(all_match_ious)
    ax.axvline(max_iou, color='purple', linestyle=':', linewidth=2, label=f'max={max_iou:.3f}')
    
    ax.set_xlabel("IoU", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    title_suffix = f" (aggregated across {len(pairwise_overlap_results)} pairs)" if len(runs) > 2 else ""
    ax.set_title(f"IoU Distribution of All Matching Pairs (n={len(all_match_ious)}){title_suffix}", fontsize=14)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"IoU Statistics:")
    print(f"  - Count: {len(all_match_ious)}")
    print(f"  - Max: {max(all_match_ious):.3f}")
    print(f"  - Mean: {np.mean(all_match_ious):.3f}")
    print(f"  - Median: {np.median(all_match_ious):.3f}")
    print(f"  - Std: {np.std(all_match_ious):.3f}")

In [ ]:
# LLM Analysis: IoU Distribution
if ENABLE_LLM_ANALYSIS and all_match_ious:
    iou_dist_analysis = analyze_iou_distribution(
        iou_values=all_match_ious,
        thresholds=[0.2, 0.3, 0.4],
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(iou_dist_analysis, "IoU Distribution")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "IoU Distribution",
        "interpretation": iou_dist_analysis.interpretation,
        "observations": iou_dist_analysis.key_observations,
    })

## 6. Trajectory Comparison

In [ ]:
# Compute trajectory correlation (already supports N runs)
layers_to_plot = [0, 6, 12, 18, 23]
corr_matrix = compute_trajectory_correlation(runs, metric="effective_rank", layers=layers_to_plot)

print("Trajectory Correlation Matrix (effective_rank):")
display(corr_matrix)

# Compute mean off-diagonal correlation
mask = ~np.eye(len(corr_matrix), dtype=bool)
off_diag = corr_matrix.values[mask]
mean_corr = np.nanmean(off_diag)
print(f"\nMean pairwise correlation: {mean_corr:.3f}")

In [ ]:
def plot_trajectory_comparison(runs, metric, layers, figsize=None):
    """Plot trajectory comparison for a metric across layers."""
    n_layers = len(layers)
    if figsize is None:
        figsize = (14, 2 * n_layers)
    fig, axes = plt.subplots(n_layers, 1, figsize=figsize, sharex=True)
    if n_layers == 1:
        axes = [axes]
    
    colors = plt.cm.tab10.colors
    
    for i, layer in enumerate(layers):
        ax = axes[i]
        
        for j, run in enumerate(runs):
            traj = extract_metric_trajectories(run.geometry_df, layers=[layer], metric=metric)
            if not traj.empty:
                seed = run.meta.get('seed', f'run{j+1}')
                ax.plot(
                    traj["step"],
                    traj["value"],
                    color=colors[j % len(colors)],
                    linewidth=1.5,
                    alpha=0.8,
                    label=f"seed={seed}"
                )
        
        ax.set_ylabel(f"Layer {layer}", fontsize=10)
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(loc="upper right", fontsize=9, ncol=min(len(runs), 4))
            ax.set_title(f"{metric} Trajectories ({len(runs)} runs)", fontsize=14)
    
    axes[-1].set_xlabel("Step", fontsize=12)
    plt.tight_layout()
    return fig, axes

In [ ]:
# Plot effective_rank trajectories
fig, axes = plot_trajectory_comparison(runs, "effective_rank", layers_to_plot)
plt.show()

In [ ]:
# Plot sv_entropy trajectories
fig, axes = plot_trajectory_comparison(runs, "sv_entropy", layers_to_plot)
plt.show()

In [ ]:
# LLM Analysis: Trajectory Comparison
if ENABLE_LLM_ANALYSIS:
    traj_analysis = analyze_trajectory_comparison(
        correlation_matrix=corr_matrix,
        metric="effective_rank",
        layers=layers_to_plot,
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(traj_analysis, "Trajectory Comparison")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Trajectory Comparison",
        "interpretation": traj_analysis.interpretation,
        "observations": traj_analysis.key_observations,
    })

## 7. Composite Event Analysis

In [ ]:
# Compute composite invariance for all pairs
pairwise_composite_results = {}

for i, j in run_pairs:
    pair_key = f"{i+1}-{j+1}"
    pair_runs = [runs[i], runs[j]]
    comp_inv = compute_composite_invariance(pair_runs, iou_threshold=0.3)
    pairwise_composite_results[pair_key] = comp_inv

# Display results
print("Composite Invariance (pairwise):\n")
for pair_key, comp_inv in pairwise_composite_results.items():
    i, j = [int(x)-1 for x in pair_key.split('-')]
    seed_i = runs[i].meta.get('seed', '?')
    seed_j = runs[j].meta.get('seed', '?')
    print(f"Pair {pair_key} (seed {seed_i} vs {seed_j}):")
    print(f"  - Composites: {comp_inv['n_composites_a']} / {comp_inv['n_composites_b']}")
    print(f"  - Matched: {comp_inv['n_matched']}")
    print(f"  - Jaccard: {comp_inv['jaccard']:.1%}")
    if comp_inv.get('strength_correlation'):
        print(f"  - Strength correlation: {comp_inv['strength_correlation']:.2f}")
    print()

# Use first pair for backward compatibility
comp_inv = list(pairwise_composite_results.values())[0] if pairwise_composite_results else {}

In [ ]:
# Plot composite events for all runs
n_runs = len(runs)
n_cols = min(n_runs, 3)
n_rows = (n_runs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), squeeze=False)

for idx, run in enumerate(runs):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row, col]
    
    comp_df = run.events_df[run.events_df["event_type"] == "change_point_composite"]
    seed = run.meta.get('seed', '?')
    
    if not comp_df.empty and "composite_strength" in comp_df.columns:
        scatter = ax.scatter(
            comp_df["step"],
            comp_df["layer"],
            c=comp_df["composite_strength"],
            s=comp_df.get("composite_n_metrics", 3) * 30,
            cmap="YlOrRd",
            alpha=0.8,
            edgecolors='black',
            linewidths=0.5
        )
        plt.colorbar(scatter, ax=ax, label="Strength")
    else:
        ax.scatter(comp_df["step"], comp_df["layer"], s=50, alpha=0.7)
    
    ax.set_xlabel("Step", fontsize=12)
    ax.set_ylabel("Layer", fontsize=12)
    ax.set_title(f"Run {idx+1}: {len(comp_df)} Composites (seed={seed})", fontsize=12)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_runs, n_rows * n_cols):
    row, col = idx // n_cols, idx % n_cols
    axes[row, col].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# LLM Analysis: Composite Events
if ENABLE_LLM_ANALYSIS:
    # Get composite counts for all runs
    composite_counts = [
        len(run.events_df[run.events_df["event_type"] == "change_point_composite"])
        for run in runs
    ]
    
    # Aggregate composite invariance across pairs
    all_comp_jaccards = [r['jaccard'] for r in pairwise_composite_results.values()]
    
    comp_analysis = analyze_composite_events(
        composite_invariance={
            **comp_inv,  # First pair details
            'n_pairs': len(pairwise_composite_results),
            'mean_jaccard_across_pairs': np.mean(all_comp_jaccards) if all_comp_jaccards else 0,
        },
        run_a_composites=composite_counts[0],
        run_b_composites=composite_counts[1] if len(composite_counts) > 1 else 0,
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(comp_analysis, "Composite Events")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Composite Events",
        "interpretation": comp_analysis.interpretation,
        "observations": comp_analysis.key_observations,
    })

## 8. Event Phase Distribution

In [ ]:
# Analyze phases for all runs
phase_results = [analyze_event_phases(run) for run in runs]

print("Phase Analysis:\n")
for i, (run, phase_data) in enumerate(zip(runs, phase_results)):
    seed = run.meta.get('seed', '?')
    print(f"Run {i+1} (seed {seed}): {phase_data['phase_counts']}")

In [ ]:
# Plot phase distribution for all runs
n_runs = len(runs)
fig, axes = plt.subplots(1, n_runs, figsize=(4 * n_runs, 4), squeeze=False)

for idx, (run, phase_data) in enumerate(zip(runs, phase_results)):
    ax = axes[0, idx]
    phases = phase_data["phase_counts"]
    seed = run.meta.get('seed', '?')
    
    if phases:
        labels = list(phases.keys())
        values = list(phases.values())
        colors = ['#2ecc71', '#3498db', '#9b59b6']  # shaping, transition, locking
        
        ax.bar(labels, values, color=colors[:len(labels)], edgecolor='black')
        ax.set_ylabel("Event Count", fontsize=12)
        ax.set_title(f"Run {idx+1} (seed {seed})", fontsize=12)
        
        # Add value labels
        for i, v in enumerate(values):
            ax.text(i, v + 0.5, str(v), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# LLM Analysis: Phase Distribution
if ENABLE_LLM_ANALYSIS:
    # Use first two runs for the existing function, but could extend to N runs
    phase_analysis = analyze_phase_distribution(
        phase_a=phase_results[0],
        phase_b=phase_results[1] if len(phase_results) > 1 else {"phase_counts": {}},
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(phase_analysis, "Phase Distribution")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Phase Distribution",
        "interpretation": phase_analysis.interpretation,
        "observations": phase_analysis.key_observations,
    })

## 9. Retention Analysis

In [ ]:
# Display retention metrics for all runs
runs_with_retention = [run for run in runs if run.retention]

if runs_with_retention:
    retention_data = {
        "Metric": ["Candidates", "Selected", "Retention Rate", "Pre-Warmup Ret.", "Post-Warmup Ret.", "Cand/Series (post)"],
    }
    
    for run in runs_with_retention:
        seed = run.meta.get('seed', '?')
        col_name = f"Run (s{seed})"
        retention_data[col_name] = [
            run.retention.n_candidates,
            run.retention.n_selected,
            f"{run.retention.retention_rate:.1%}",
            f"{run.retention.pre_retention_rate:.1%}",
            f"{run.retention.post_retention_rate:.1%}",
            f"{run.retention.mean_candidates_post_per_series:.1f}",
        ]
    
    retention_df = pd.DataFrame(retention_data)
    display(retention_df)
    
    # Compute density ratios between pairs
    if len(runs_with_retention) >= 2:
        print("\nDensity Ratios (pairwise):")
        for i in range(len(runs_with_retention)):
            for j in range(i + 1, len(runs_with_retention)):
                r1, r2 = runs_with_retention[i].retention, runs_with_retention[j].retention
                ratio = r1.mean_candidates_post_per_series / max(r2.mean_candidates_post_per_series, 0.1)
                s1 = runs_with_retention[i].meta.get('seed', '?')
                s2 = runs_with_retention[j].meta.get('seed', '?')
                print(f"  seed {s1} / seed {s2}: {ratio:.2f}x")
else:
    print("Retention data not available for any runs.")

In [ ]:
# LLM Analysis: Retention
if ENABLE_LLM_ANALYSIS and len(runs_with_retention) >= 2:
    retention_analysis = analyze_retention(
        retention_a={
            "n_candidates": runs_with_retention[0].retention.n_candidates,
            "n_selected": runs_with_retention[0].retention.n_selected,
            "retention_rate": f"{runs_with_retention[0].retention.retention_rate:.1%}",
            "pre_retention_rate": f"{runs_with_retention[0].retention.pre_retention_rate:.1%}",
            "post_retention_rate": f"{runs_with_retention[0].retention.post_retention_rate:.1%}",
            "mean_candidates_post_per_series": f"{runs_with_retention[0].retention.mean_candidates_post_per_series:.1f}",
        },
        retention_b={
            "n_candidates": runs_with_retention[1].retention.n_candidates,
            "n_selected": runs_with_retention[1].retention.n_selected,
            "retention_rate": f"{runs_with_retention[1].retention.retention_rate:.1%}",
            "pre_retention_rate": f"{runs_with_retention[1].retention.pre_retention_rate:.1%}",
            "post_retention_rate": f"{runs_with_retention[1].retention.post_retention_rate:.1%}",
            "mean_candidates_post_per_series": f"{runs_with_retention[1].retention.mean_candidates_post_per_series:.1f}",
        },
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(retention_analysis, "Retention Metrics")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Retention Metrics",
        "interpretation": retention_analysis.interpretation,
        "observations": retention_analysis.key_observations,
    })

## 10. Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nRuns compared: {len(runs)}")
print(f"Seeds: {', '.join(str(s) for s in run_seeds)}")

print(f"\n1. SIGNAL INVARIANCE (Trajectory Level)")
print(f"   - Mean trajectory correlation: {mean_corr:.3f}")

print(f"\n2. EVENT INVARIANCE (Selected Peaks)")
print(f"   - Common events (strict +/-{STEP_TOLERANCE}): {len(common_events)}")

# Aggregate IoU metrics across all pairs
mean_iou_jaccard = np.mean(all_jaccards) if all_jaccards else 0
mean_comp_jaccard = np.mean([r['jaccard'] for r in pairwise_composite_results.values()]) if pairwise_composite_results else 0

print(f"   - Mean IoU Jaccard (tau={IOU_THRESHOLD}): {mean_iou_jaccard:.1%}")
print(f"   - Mean Composite Jaccard: {mean_comp_jaccard:.1%}")

if len(all_jaccards) > 1:
    print(f"   - IoU Jaccard range: {np.min(all_jaccards):.1%} - {np.max(all_jaccards):.1%}")

print(f"\n3. IoU DISTRIBUTION")
if all_match_ious:
    print(f"   - Max IoU: {max(all_match_ious):.3f}")
    print(f"   - Median IoU: {np.median(all_match_ious):.3f}")

print(f"\n4. INTERPRETATION")
if mean_corr > 0.9 and mean_iou_jaccard < 0.3:
    print("   Signal-Event Discrepancy: High trajectory correlation but low event overlap.")
    print("   This indicates selection sensitivity, not signal difference.")
elif mean_corr > 0.8:
    print("   Runs show consistent learning dynamics at the trajectory level.")

In [ ]:
# LLM Analysis: Final Summary
if ENABLE_LLM_ANALYSIS:
    # Get max IoU from distribution if available
    iou_max = max(all_match_ious) if all_match_ious else None
    
    summary_analysis = analyze_summary(
        trajectory_correlation=mean_corr if not np.isnan(mean_corr) else 0.0,
        iou_jaccard=mean_iou_jaccard,
        composite_jaccard=mean_comp_jaccard,
        iou_threshold=IOU_THRESHOLD,
        n_common_strict=len(common_events),
        iou_max=iou_max,
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    display_analysis(summary_analysis, "Final Summary")
    
    # Store for overall synthesis
    SECTION_ANALYSES.append({
        "section_name": "Summary Statistics",
        "interpretation": summary_analysis.interpretation,
        "observations": summary_analysis.key_observations,
    })

## 11. Overall LLM Synthesis

This section synthesizes all the individual section analyses into a comprehensive qualitative assessment.

In [ ]:
# Overall LLM Synthesis - Combines all section analyses
if ENABLE_LLM_ANALYSIS and SECTION_ANALYSES:
    print(f"Synthesizing {len(SECTION_ANALYSES)} section analyses...\n")
    
    overall_analysis = analyze_overall(
        section_summaries=SECTION_ANALYSES,
        n_runs=len(runs),
        run_seeds=run_seeds,
        backend=LLM_BACKEND,
        model=LLM_MODEL,
    )
    
    # Display with special formatting for the overall synthesis
    from IPython.display import Markdown, display as ipy_display
    
    ipy_display(Markdown("---"))
    ipy_display(Markdown("## Overall Qualitative Assessment"))
    ipy_display(Markdown(""))
    
    if overall_analysis.error:
        ipy_display(Markdown(f"**Error:** {overall_analysis.error}"))
    else:
        if overall_analysis.interpretation:
            ipy_display(Markdown(f"### Summary\n\n{overall_analysis.interpretation}"))
        
        if overall_analysis.key_observations:
            ipy_display(Markdown("\n### Key Findings"))
            for i, obs in enumerate(overall_analysis.key_observations, 1):
                ipy_display(Markdown(f"{i}. {obs}"))
        
        if overall_analysis.questions:
            ipy_display(Markdown("\n### Recommendations & Open Questions"))
            for q in overall_analysis.questions:
                ipy_display(Markdown(f"- {q}"))
    
    ipy_display(Markdown("---"))
else:
    if not ENABLE_LLM_ANALYSIS:
        print("LLM analysis disabled. Set ENABLE_LLM_ANALYSIS = True to enable.")
    else:
        print("No section analyses collected. Run all cells above first.")